In [2]:
!pip install -q sentence-transformers datasets pandas faiss-gpu gradio requests

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import faiss
import random
import textwrap
import requests
import re
import torch
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
from datasets import load_dataset
import plotly.graph_objects as go
import plotly.io as pio
import gradio as gr
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 13.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.9 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inc

In [3]:
print("Loading Koziev's paraphrases from HuggingFace")
dataset = load_dataset("inkoziev/paraphrases", split="train")

train_examples = []
limit = len(dataset)

for i in range(limit):
    row = dataset[i]
    
    texts = []
    for val in row.values():
        if isinstance(val, str):
            texts.append(val)
        elif isinstance(val, list) and all(isinstance(x, str) for x in val):
            texts.extend(val)
            
    if len(texts) >= 2:
        text_a, text_b = texts[0], texts[1]
        if len(text_a) > 10 and len(text_b) > 10:
            train_examples.append(InputExample(texts=[text_a, text_b]))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=64)

print(f"Prepared {len(train_examples)} semantic pairs for training")

Loading Koziev's paraphrases from HuggingFace


README.md: 0.00B [00:00, ?B/s]

paraphrases.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/17941 [00:00<?, ? examples/s]

Prepared 17399 semantic pairs for training


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device detected for training: {device}")

model = SentenceTransformer("sentence-transformers/LaBSE", device=device)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

print("Starting training of LaBSE model on Koziev's paraphrases")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="./LaBSE-koziev"
)
print("Fine-tuning completed")

Device detected for training: cuda


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Starting training of LaBSE model on Koziev's paraphrases


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning completed


In [5]:
print("Sourcing Russian proverbs from Wikiquote API")

cat = "Русские_пословицы"
proverbs = set()

headers = {
    'User-Agent': 'SemanticProverbsExplorer/1.0 (educational project; contact: student@kaggle.com)'
}

params = {
    "action": "query",
    "prop": "revisions",
    "rvprop": "content",
    "titles": cat,
    "format": "json",
    "utf8": 1
}

response = requests.get(
    "https://ru.wikiquote.org/w/api.php", 
    params=params, 
    headers=headers,
    timeout=15
)
response.raise_for_status()
res = response.json()

pages = res['query']['pages']
page_id = list(pages.keys())[0]

content = pages[page_id].get('revisions', [{}])[0].get('*', '')

for line in content.split('\n'):
    if line.strip().startswith('*'):
        line = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', line)
        line = re.sub(r'\{\{.*?\}\}', '', line)
        line = re.sub(r'<[^>]+>', '', line)
        line = line.strip('* \'"\t ')
        
        if 4 <= len(line.split()) <= 20 and not '=' in line and not 'Пословицы' in line:
            proverbs.add(line)

proverbs_list = list(proverbs)
print(f"Extraction complete. Total unique Russian proverbs: {len(proverbs_list)}")

Sourcing Russian proverbs from Wikiquote API
Extraction complete. Total unique Russian proverbs: 3556


In [6]:
print("Encoding proverbs database with LaBSE")
corpus_emb = model.encode(proverbs_list, normalize_embeddings=True, show_progress_bar=True)
corpus_emb = np.array(corpus_emb, dtype=np.float32)

embedding_dim = corpus_emb.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(corpus_emb)

_ = model.encode(["warming"], normalize_embeddings=True)

print("Clustering and computing t-SNE coordinates")
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(corpus_emb)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
proverbs_2d = tsne.fit_transform(corpus_emb)

base_df = pd.DataFrame({
    'x': proverbs_2d[:, 0],
    'y': proverbs_2d[:, 1],
    'Пословица': proverbs_list,
    'Категория': [f"Группа {c+1}" for c in cluster_labels]
})

base_df['Пословица_Wrapped'] = base_df['Пословица'].apply(
    lambda x: "<br>".join(textwrap.wrap(x, width=45))
)

all_categories = ["Все", "—"] + sorted(base_df['Категория'].unique().tolist())

palette = {cat: col for cat, col in zip(
    sorted(base_df['Категория'].unique()),
    ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
)}

x_min_all, x_max_all = base_df['x'].min(), base_df['x'].max()
y_min_all, y_max_all = base_df['y'].min(), base_df['y'].max()

x_span_all = x_max_all - x_min_all if x_max_all != x_min_all else 1.0
y_span_all = y_max_all - y_min_all if y_max_all != y_min_all else 1.0

global_padding = 0.10
global_zoom_range = {
    'x_range': [x_min_all - x_span_all * global_padding, x_max_all + x_span_all * global_padding],
    'y_range': [y_min_all - y_span_all * global_padding, y_max_all + y_span_all * global_padding]
}

Encoding proverbs database with LaBSE


Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Clustering and computing t-SNE coordinates


In [7]:
sample_inputs = [
    "нужно усердно и долго трудиться, чтобы получить хороший результат",
    "настоящий верный друг познается только в трудной ситуации",
    "когда люди слишком сильно спешат, они совершают глупые ошибки",
    "надо ценить и любить своих родителей, пока они рядом",
    "если делать дело вместе и дружно, то любая работа покажется легкой",
    "человек, который слишком много разговаривает, часто ошибается",
    "внешний вид обманчив, нужно оценивать людей по уму и делам",
    "не стоит бояться трудностей, иначе никогда ничего не добьешься",
    "если совершил плохой поступок, правда все равно рано или поздно вскроется",
    "нельзя вернуть то, что уже было сказано или сделано в прошлом"
]

def get_random_example():
    return random.choice(sample_inputs)

def make_figure(query_point=None, highlight_cluster="Все",
                highlight_indices=None, zoom_range=None):
    fig = go.Figure()

    if zoom_range is None:
        zoom_range = global_zoom_range

    if highlight_indices is not None:
        bright_mask = base_df.index.isin(highlight_indices)
    elif highlight_cluster != "Все" and highlight_cluster != "—":
        bright_mask = base_df['Категория'] == highlight_cluster
    else:
        bright_mask = np.ones(len(base_df), dtype=bool)

    dim_df = base_df[~bright_mask]
    bright_df = base_df[bright_mask]

    if not dim_df.empty:
        fig.add_trace(go.Scatter(
            x=dim_df['x'], y=dim_df['y'],
            mode='markers',
            marker=dict(
                color='#cbd5e1',
                size=9,
                opacity=0.15,
                line=dict(width=0.5, color='white')
            ),
            hoverinfo='skip', 
            showlegend=False
        ))

    if not bright_df.empty:
        fig.add_trace(go.Scatter(
            x=bright_df['x'], y=bright_df['y'],
            mode='markers',
            marker=dict(
                color=bright_df['Категория'].map(palette),
                size=10,
                opacity=0.9,
                line=dict(width=0.8, color='white')
            ),
            text=bright_df['Пословица_Wrapped'],
            hoverinfo='text',
            showlegend=False
        ))

    if query_point is not None:
        fig.add_trace(go.Scatter(
            x=[query_point['x']],
            y=[query_point['y']],
            mode='markers',
            marker=dict(
                symbol='star',
                size=22,
                color='#e11d48',
                line=dict(width=2, color='#ffffff')
            ),
            text=query_point['hover_text'],
            hoverinfo='text',
            showlegend=False
        ))

    fig.update_layout(
        font_family="Arial, sans-serif",
        margin=dict(l=5, r=5, t=5, b=5),
        plot_bgcolor='rgba(248, 250, 252, 0.8)',
        paper_bgcolor='rgba(0,0,0,0)',
        hovermode='closest', 
        hoverdistance=45,    
        modebar=dict(
            remove=[
                "autoScale2d", "autoscale", "lasso", "lasso2d", "pan", "pan2d", 
                "reset", "resetScale2d", "select", "select2d", "toImage", 
                "toggleHover", "toggleSpikelines", "zoom", "zoom2d", "zoomin", "zoomout"
            ]
        ),
        hoverlabel=dict(     
            bgcolor="#1e293b",
            font_size=13,
            font_color="#ffffff",
            font_family="Arial",
            bordercolor="#334155"
        ),
        xaxis=dict(
            showgrid=False, zeroline=False, showticklabels=False, showline=False, title=None,
            fixedrange=True
        ),
        yaxis=dict(
            showgrid=False, zeroline=False, showticklabels=False, showline=False, title=None,
            fixedrange=True,
            scaleanchor="x",
            scaleratio=1,
            range=zoom_range['y_range']
        ),
        showlegend=False,
        dragmode=False
    )

    return fig

def initial_map():
    return make_figure()

def unified_search(query, current_highlight):
    if not query.strip():
        return initial_map(), """
        <div style="color: #64748b; padding: 2rem; text-align: center; border: 2px dashed #cbd5e1; border-radius: 16px;">
            <p>Введите запрос, чтобы увидеть подходящие пословицы</p>
        </div>
        """

    query_emb = model.encode([query], normalize_embeddings=True)
    query_emb = np.array(query_emb, dtype=np.float32)

    top_k_html = 5
    top_k_map = 20
    
    distances_map, indices_map = index.search(query_emb, top_k_map)
    map_indices = indices_map[0]

    top_indices = map_indices[:top_k_html]
    distances_html = distances_map[0][:top_k_html]
    
    neighbor_coords_5 = proverbs_2d[top_indices]
    weights = np.exp(distances_html * 12)
    weights /= np.sum(weights)
    q_x = np.sum(neighbor_coords_5[:, 0] * weights)
    q_y = np.sum(neighbor_coords_5[:, 1] * weights)

    wrapped_query = "<br>".join(textwrap.wrap(f"Запрос: '{query}'", width=45))
    query_point = {
        'x': q_x,
        'y': q_y,
        'hover_text': wrapped_query
    }

    neighbor_coords_20 = proverbs_2d[map_indices]
    all_x = list(neighbor_coords_20[:, 0]) + [q_x]
    all_y = list(neighbor_coords_20[:, 1]) + [q_y]
    
    x_min, x_max = min(all_x), max(all_x)
    y_min, y_max = min(all_y), max(all_y)
    
    x_span = x_max - x_min if x_max != x_min else 1.0
    y_span = y_max - y_min if y_max != y_min else 1.0
    
    padding = 0.25
    zoom_range = {
        'x_range': [x_min - x_span * padding, x_max + x_span * padding],
        'y_range': [y_min - y_span * padding, y_max + y_span * padding]
    }

    fig = make_figure(query_point,
                      highlight_cluster="Все",
                      highlight_indices=map_indices,
                      zoom_range=zoom_range)

    html_results = f"""
    <div style="margin-top: 0.5rem;">
        <div style="margin-bottom: 1rem;">
            <h4 style="color: #0f172a; margin: 0;">Ближайшие пословицы</h4>
            <span style="background: #f1f5f9; color: #334155; padding: 2px 12px; border-radius: 999px; font-size: 0.8rem;">{top_k_html} результатов</span>
        </div>
    """
    for i in range(top_k_html):
        idx = top_indices[i]
        score = distances_html[i]
        html_results += f"""
        <div class="proverb-card" style="margin-bottom: 10px; padding: 16px; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; box-shadow: 0 1px 4px rgba(0,0,0,0.02);">
            <p style="margin: 0; font-size: 15px; font-weight: 550; color: #1e293b; line-height: 1.4;">{proverbs_list[idx]}</p>
            <div style="display: flex; justify-content: space-between; align-items: center; margin-top: 8px;">
                <span style="font-size: 12px; color: #64748b;">Совпадение</span>
                <span style="font-weight: 600; font-size: 13px; color: #0f172a; background: #f8fafc; padding: 2px 10px; border-radius: 6px;">{score:.3f}</span>
            </div>
        </div>
        """
    html_results += "</div>"

    return fig, html_results, "—"

def update_highlight(selected_cluster):
    if selected_cluster == "—":
        return gr.update(), gr.update()
        
    if selected_cluster == "Все":
        fig = make_figure(highlight_cluster="Все", zoom_range=global_zoom_range)
        empty_html = """
        <div style="color: #64748b; padding: 2rem; text-align: center; border: 2px dashed #cbd5e1; border-radius: 16px;">
            <p>Введите запрос, чтобы увидеть подходящие пословицы</p>
            <p style="font-size: 0.9rem; margin-top: 8px;">Результаты появятся здесь</p>
        </div>
        """
        return fig, empty_html
        
    fig = make_figure(highlight_cluster=selected_cluster, zoom_range=global_zoom_range)
    
    sub_df = base_df[base_df['Категория'] == selected_cluster]
    sampled = sub_df.sample(min(5, len(sub_df)))
    
    html_results = f"""
    <div style="margin-top: 0.5rem;">
        <div style="margin-bottom: 1rem;">
            <h4 style="color: #0f172a; margin: 0;">Пословицы из {selected_cluster}</h4>
            <span style="background: #f1f5f9; color: #334155; padding: 2px 12px; border-radius: 999px; font-size: 0.8rem;">Случайная подборка</span>
        </div>
    """
    for _, row in sampled.iterrows():
        html_results += f"""
        <div class="proverb-card" style="margin-bottom: 10px; padding: 16px; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; box-shadow: 0 1px 4px rgba(0,0,0,0.02);">
            <p style="margin: 0; font-size: 15px; font-weight: 550; color: #1e293b; line-height: 1.4;">{row['Пословица']}</p>
        </div>
        """
    html_results += "</div>"
    
    return fig, html_results

def search_with_example(current_highlight):
    random_query = get_random_example()
    fig, html_results, highlight = unified_search(random_query, current_highlight)
    return random_query, fig, html_results, highlight

custom_css = """
    .proverb-card:hover {
        box-shadow: 0 6px 20px rgba(0,0,0,0.06) !important;
        border-color: #cbd5e1 !important;
    }
    #search_button, #clear_button {
        white-space: nowrap;
        padding: 0.6rem 1.5rem;
        font-size: 14px;
        width: 100%;
    }
    .gradio-container {
        max-width: 1400px !important;
        margin: auto;
    }
    #cluster_dropdown {
        margin-bottom: 12px;
    }
    #example_button {
        margin-top: 4px !important;
        margin-bottom: 4px !important;
        width: 100%;
    }
    .plotly-graph-div {
        height: 680px !important;
    }
    .modebar-container, .modebar, .watermark, .plotly-notifier {
        display: none !important;
    }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="slate"), css=custom_css) as demo:
    gr.Markdown("# Семантический анализ русских пословиц")
    gr.Markdown("Опишите ситуацию – система найдёт пословицы, близкие по смыслу, и покажет их на карте.")

    with gr.Row():
        with gr.Column(scale=7): 
            map_plot = gr.Plot(show_label=False)
        with gr.Column(scale=3, min_width=300): 
            gr.Markdown("### Поиск по смыслу")
            query_input = gr.Textbox(
                show_label=False, 
                placeholder="Например: нужно усердно работать, чтобы получить результат",
                lines=3
            )
            
            example_btn = gr.Button("Случайный пример", variant="secondary", size="sm", elem_id="example_button")
            
            with gr.Row():
                btn = gr.Button("Поиск", variant="primary", elem_id="search_button")
                clear_btn = gr.ClearButton(query_input, value="Очистить", variant="secondary", size="sm", elem_id="clear_button")

            gr.Markdown("### Смысловые группы")
            cluster_dropdown = gr.Dropdown(
                choices=all_categories,
                value="Все",
                show_label=False,         
                interactive=True,
                filterable=False,         
                allow_custom_value=False, 
                elem_id="cluster_dropdown"
            )
            output_html = gr.HTML()

    demo.load(
        fn=lambda: (initial_map(), """
        <div style="color: #64748b; padding: 2rem; text-align: center; border: 2px dashed #cbd5e1; border-radius: 16px;">
            <p>Введите запрос, чтобы увидеть подходящие пословицы</p>
            <p style="font-size: 0.9rem; margin-top: 8px;">Результаты появятся здесь</p>
        </div>
        """, "Все"),
        outputs=[map_plot, output_html, cluster_dropdown]
    )

    btn.click(
        fn=unified_search,
        inputs=[query_input, cluster_dropdown],
        outputs=[map_plot, output_html, cluster_dropdown]
    )
    query_input.submit(
        fn=unified_search,
        inputs=[query_input, cluster_dropdown],
        outputs=[map_plot, output_html, cluster_dropdown]
    )
    
    example_btn.click(
        fn=search_with_example,
        inputs=[cluster_dropdown],
        outputs=[query_input, map_plot, output_html, cluster_dropdown]
    )
    
    cluster_dropdown.change(
        fn=update_highlight,
        inputs=[cluster_dropdown],
        outputs=[map_plot, output_html]
    )

demo.launch(share=True, inline=False)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b5993d846adb2f1186.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
